# 06 — Visualizaciones

Solo lee los CSV generados en el cuaderno 05 y dibuja. **No recalcula nada.** Todas las figuras usan la misma paleta Tab20 para sectores GICS y se guardan en `outputs/figuras/`.

In [1]:
# Lee de PATHS["clusters_out"] los artefactos generados en NB05.
# matplotlib puro. Paleta Tab20 fija para sectores GICS.
# Cada figura se guarda en outputs/figuras/.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- Rutas ---
PROJECT_ROOT = Path.cwd()
OUT = PROJECT_ROOT / "outputs"
FIG = OUT / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

VENTANA_VIZ = 2023  # ventana de referencia para figuras de composicion

# --- Estilo sobrio y coherente ---
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
})

# --- Paleta GICS fija (Tab20). Mismo color de sector en TODAS las figuras. ---
SECTORES_GICS = [
    "Information Technology", "Industrials", "Financials", "Health Care",
    "Consumer Discretionary", "Real Estate", "Consumer Staples", "Utilities",
    "Materials", "Energy", "Communication Services",
]
_tab20 = plt.get_cmap("tab20").colors
COLOR_GICS = {s: _tab20[i] for i, s in enumerate(SECTORES_GICS)}
COLOR_REPR = {"tfidf": "#1b6ca8", "finbert": "#c0392b", "sbert": "#27ae60"}
NOMBRE_REPR = {"tfidf": "TF-IDF", "finbert": "FinBERT", "sbert": "SBERT"}

## Figura 1 — Proceso de selección de los modelos finalistas

In [2]:
# FIGURA 1 — Proceso de selección de los modelos finalistas
def figura_embudo():
    df = pd.read_csv(OUT / "Embudo_Filtro.csv")
    etapas = ["etapa_0_total", "etapa_1_estabilidad",
              "etapa_2_calidad_relativa", "etapa_3_finalista"]
    labels = ["Rejilla\ncompleta", "Etapa 1\nestabilidad",
              "Etapa 2\ncalidad relativa", "Finalista"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(etapas)); w = 0.25
    for j, r in enumerate(["tfidf", "finbert", "sbert"]):
        row = df[df.representation == r].iloc[0]
        vals = [row[e] for e in etapas]
        ax.bar(x + (j - 1) * w, vals, w, label=NOMBRE_REPR[r], color=COLOR_REPR[r])
        for xi, v in zip(x + (j - 1) * w, vals):
            ax.text(xi, v + 2, str(int(v)), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Nº de configuraciones")
    ax.legend(frameon=False)
    fig.savefig(FIG / "fig1_embudo_filtro.png")
    plt.close(fig)
    print("fig1_embudo_filtro.png")


## Figura 2 — Comparación de clasificaciones en 2D (UMAP)

In [3]:
# FIGURA 2 — UMAP 2D ilustrativo (panel emparejado HDBSCAN vs GICS)

def figura_umap_2d():
    df = pd.read_csv(OUT / "UMAP2D_TFIDF.csv")
    df = df.dropna(subset=["x", "y"])

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(12.5, 5.5), sharex=True, sharey=True)

    # Panel A: coloreado por cluster HDBSCAN
    clusters = sorted(df["cluster_label"].unique())
    cmap_cl = plt.get_cmap("tab20", max(len(clusters), 3))
    for i, cl in enumerate(clusters):
        sub = df[df["cluster_label"] == cl]
        axA.scatter(sub["x"], sub["y"], s=14, color=cmap_cl(i),
                    label=f"C{cl}", alpha=0.85, edgecolor="white", linewidth=0.3)
    axA.set_title("Panel A — Coloreado por clúster HDBSCAN (TF-IDF)")
    axA.legend(loc="best", fontsize=7, ncol=2, frameon=False, markerscale=1.2)

    # Panel B: coloreado por sector GICS
    df_g = df.dropna(subset=["gics_sector"])
    for sector in SECTORES_GICS:
        sub = df_g[df_g["gics_sector"] == sector]
        if sub.empty:
            continue
        axB.scatter(sub["x"], sub["y"], s=14, color=COLOR_GICS[sector],
                    label=sector, alpha=0.85, edgecolor="white", linewidth=0.3)
    axB.set_title("Panel B — Coloreado por sector GICS")
    axB.legend(loc="best", fontsize=7, ncol=1, frameon=False, markerscale=1.2,
               bbox_to_anchor=(1.02, 1))

    for ax in (axA, axB):
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
        ax.set_xticks([]); ax.set_yticks([])

    fig.savefig(FIG / "fig2_umap_2d.png", bbox_inches="tight")
    plt.close(fig)
    print("fig2_umap_2d.png")


## Figura 3 — Tamaños de clúster

In [4]:
# FIGURA 3 — Tamaños de cluster
def figura_tamanos():
    df = pd.read_csv(OUT / "Tamanos_Clusteres_WF.csv")
    df = df[df.year_T == VENTANA_VIZ]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for r in ["tfidf", "finbert", "sbert"]:
        sub = df[df.representation == r].sort_values("n_empresas", ascending=False)
        ax.plot(range(1, len(sub) + 1), sub["n_empresas"].values,
                marker="o", label=NOMBRE_REPR[r], color=COLOR_REPR[r])
    ax.set_xlabel("Rango del clúster (mayor → menor)")
    ax.set_ylabel("Nº de empresas en el clúster")
    ax.legend(frameon=False)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    fig.savefig(FIG / "fig3_tamanos.png")
    plt.close(fig)
    print("fig3_tamanos.png")


## Figura 4 — Matriz de confusión clúster TF-IDF → GICS

Aquí va normalizado por filas (P(sector|clúster)).

In [5]:
# FIGURA 4 — Matriz de confusion cluster -> GICS (heatmap normalizado por filas)

def figura_confusion_tfidf(normalizar="filas"):
    df = pd.read_csv(OUT / "Composicion_Clusteres_WF.csv")
    df = df[(df.representation == "tfidf") & (df.year_T == VENTANA_VIZ)].dropna(subset=["gics_sector"])
    ct = pd.crosstab(df["cluster_label"], df["gics_sector"])
    cols = [c for c in SECTORES_GICS if c in ct.columns]
    ct = ct[cols]
    if normalizar == "filas":
        mat = ct.div(ct.sum(axis=1), axis=0)
    else:
        mat = ct.div(ct.sum(axis=0), axis=1)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    im = ax.imshow(mat.values, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(mat.index))); ax.set_yticklabels([f"C{c}" for c in mat.index], fontsize=8)
    ax.set_xlabel("Sector GICS"); ax.set_ylabel("Clúster TF-IDF")
    fig.colorbar(im, ax=ax, shrink=0.8, label="Proporción")
    fig.savefig(FIG / "fig4_confusion_tfidf.png")
    plt.close(fig)
    ct.to_csv(OUT / "Crosstab_TFIDF_GICS.csv")
    print("fig4_confusion_tfidf.png | Crosstab_TFIDF_GICS.csv")
    return ct


## Figura 5 — Cuadrante geometría (DBCV) vs economía (reducción RMSE)

Tres finalistas, sin línea de tendencia. Eje Y invertido: *mejor que GICS* arriba.

In [6]:
# FIGURA 5 — Cuadrante geometria (DBCV) vs economia (reduccion RMSE)
# (el eje Y esta invertido, mejor queda arriba).

def figura_cuadrante():
    df = pd.read_csv(OUT / "Cuadrante_Geom_vs_Econ.csv")
    fig, ax = plt.subplots(figsize=(7.5, 6))
    ax.axhline(0, color="grey", lw=1, ls="--")
    for _, r in df.iterrows():
        c = COLOR_REPR[r["representation"]]
        ax.scatter(r["dbcv"], r["reduccion_pct"], s=180, color=c, zorder=3,
                   edgecolor="black", linewidth=0.5)
        ax.annotate(NOMBRE_REPR[r["representation"]],
                    (r["dbcv"], r["reduccion_pct"]),
                    textcoords="offset points", xytext=(10, 6), fontsize=10, weight="bold")
    ax.set_xlabel("DBCV medio")
    ax.set_ylabel("Reducción del RMSE (%)")
    ax.invert_yaxis()
    # Eje X comienza en 0
    xmax = max(df["dbcv"].max() * 1.10, 0.05)
    ax.set_xlim(0, xmax)
    fig.savefig(FIG / "fig5_cuadrante.png")
    plt.close(fig)
    print("fig5_cuadrante.png")


## Figura 6 — Heatmap interanual de robustez

In [7]:
# FIGURA 6 — Heatmap interanual de robustez (reduccion RMSE por año)
def figura_interanual():
    df = pd.read_csv(OUT / "Resultados_IndiceUnico_WF_anual.csv")
    df["reduccion_pct"] = (df["rmse_hdbscan"] - df["rmse_gics"]) / df["rmse_gics"] * 100
    piv = df.pivot_table(index="nombre_corto", columns="year_eval", values="reduccion_pct")
    fig, ax = plt.subplots(figsize=(7, 3.6))
    vmax = np.nanmax(np.abs(piv.values))
    im = ax.imshow(piv.values, cmap="RdYlGn_r", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f"{int(c)}" + ("*" if int(c) == 2024 else "") for c in piv.columns])
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels([i.split("_")[0].upper() for i in piv.index], fontsize=8)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:+.1f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Reducción %")
    fig.savefig(FIG / "fig6a_interanual.png")
    plt.close(fig)
    print("fig6a_interanual.png")


## Figura 7 — Violines del bootstrap con IC95 BCa

In [8]:
# FIGURA 7 — Violines del bootstrap (2000 replicas) con IC95 BCa

def figura_violines_bootstrap():
    reps = pd.read_csv(OUT / "Bootstrap_Replicas_RMSE.csv")
    bca = pd.read_csv(OUT / "Bootstrap_Reduccion_RMSE.csv").set_index("nombre_corto")
    cols = list(reps.columns)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    parts = ax.violinplot([reps[c].values for c in cols], vert=False,
                          showmeans=False, showextrema=False)
    for i, c in enumerate(cols):
        repr_name = c.split("_")[0]
        parts["bodies"][i].set_facecolor(COLOR_REPR.get(repr_name, "grey"))
        parts["bodies"][i].set_alpha(0.5)
        if c in bca.index:
            lo, hi = bca.loc[c, "ic95_bca_inf"], bca.loc[c, "ic95_bca_sup"]
            med = bca.loc[c, "reduccion_mediana_pct"]
            ax.plot([lo, hi], [i + 1, i + 1], color="black", lw=2)
            ax.plot(med, i + 1, "o", color="black", ms=5)
    ax.axvline(0, color="red", lw=1.5, ls="--", zorder=0)
    ax.set_yticks(range(1, len(cols) + 1))
    ax.set_yticklabels([NOMBRE_REPR.get(c.split("_")[0], c) for c in cols])
    ax.set_xlabel("Reducción del RMSE (%)")
    fig.savefig(FIG / "fig6b_violines_bootstrap.png")
    plt.close(fig)
    print("fig6b_violines_bootstrap.png")


## Figura 8 — Top palabras con más peso en cada cluster del TF-IDF

In [9]:
# FIGURA 8 — Perfil lexico (top terminos TF-IDF por cluster)

def figura_top_terminos_tfidf():
    df = pd.read_csv(OUT / "TopTerminos_TFIDF.csv")
    clusters = (df.groupby("cluster_label")["n_empresas_T"].first()
                  .sort_values(ascending=False).index.tolist())

    n = len(clusters)
    ncols = 3
    nrows = int(np.ceil(n / ncols))

    # Solo eliminamos bordes superflous (NO cambiamos la fuente; usa la
    # global definida en plt.rcParams para coherencia con el resto).
    with plt.rc_context({
        "axes.spines.left": False,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.bottom": True,
    }):
        fig, axes = plt.subplots(nrows, ncols, figsize=(11.5, 2.8 * nrows))
        axes = np.array(axes).reshape(-1)

        # Gradiente monocromo basado en COLOR_REPR["tfidf"]:
        # mas claro = menor peso, mas oscuro = mayor peso.
        base = COLOR_REPR["tfidf"]
        cmap = plt.cm.colors.LinearSegmentedColormap.from_list(
            "tfidf_grad", ["#cfdfe9", base])

        for ax, cl in zip(axes, clusters):
            sub = (df[df["cluster_label"] == cl]
                   .sort_values("rango", ascending=False))   # mayor peso arriba
            n_emp = int(sub["n_empresas_T"].iloc[0])
            pesos = sub["peso_medio"].values
            terminos = sub["termino"].values

            # Normalizar pesos a [0.3, 1.0] para el gradiente
            if pesos.max() > pesos.min():
                norm = (pesos - pesos.min()) / (pesos.max() - pesos.min())
            else:
                norm = np.ones_like(pesos)
            colors = [cmap(0.3 + 0.7*v) for v in norm]

            bars = ax.barh(terminos, pesos, color=colors, edgecolor="white",
                           linewidth=0.5, height=0.72)

            # Grid solo vertical, sutil
            ax.grid(axis="x", linestyle="-", linewidth=0.4, alpha=0.35, zorder=0)
            ax.set_axisbelow(True)

            # Valor numerico al final de cada barra
            xmax_local = pesos.max()
            for bar, val in zip(bars, pesos):
                ax.text(bar.get_width() + xmax_local*0.02,
                        bar.get_y() + bar.get_height()/2,
                        f"{val:.3f}",
                        va="center", ha="left", fontsize=7.5, color="#444444")

            ax.set_title(f"Clúster {cl}  ·  n = {n_emp}",
                         fontsize=10, loc="left", pad=8, color="#222222")
            ax.set_xlabel("Peso TF-IDF medio", fontsize=8, color="#555555")
            ax.tick_params(axis="y", labelsize=9, length=0)
            ax.tick_params(axis="x", labelsize=7, colors="#555555")
            ax.spines["bottom"].set_color("#888888")
            ax.spines["bottom"].set_linewidth(0.6)
            ax.set_xlim(0, xmax_local * 1.18)

        for ax in axes[len(clusters):]:
            ax.set_visible(False)

        fig.tight_layout(h_pad=2.2, w_pad=2.5)
        fig.savefig(FIG / "fig7_top_terminos_tfidf.png")
        plt.close(fig)
    print("fig7_top_terminos_tfidf.png")


## Ejecutar todas las figuras

In [10]:
figura_embudo()
figura_umap_2d()
figura_tamanos()
figura_confusion_tfidf(normalizar="filas")
figura_cuadrante()
figura_interanual()
figura_violines_bootstrap()
figura_top_terminos_tfidf()
print("Figuras en:", FIG)


fig1_embudo_filtro.png
fig2_umap_2d.png
fig3_tamanos.png
fig4_confusion_tfidf.png | Crosstab_TFIDF_GICS.csv
fig5_cuadrante.png
fig6a_interanual.png
fig6b_violines_bootstrap.png
fig7_top_terminos_tfidf.png
Figuras en: C:\Users\diego\TFG\outputs\figuras


Para tener una idea más clara de la participación de cada grupo formado por el modelo TF-IDF sobre cada sector GICS, hacemos un pequeño cálculo final que nos aporta los valores concretos de: 1) Aquellos grupos que tienen concentrado su peso en más de un 80% en un único sector 2) Aquellos grupos que requieren de tres o más sectores para concentrar el 90% de la fila. Por tanto, si el cluster no aparece en los resutlados es porque ha agrupado empresas que pertenecen en esencia a dos sectores GICS.

In [11]:
import pandas as pd

# Reconstruir la matriz normalizada por filas (idéntica a la de la figura)
df = pd.read_csv("outputs/Composicion_Clusteres_WF.csv")
df = df[(df["representation"] == "tfidf") & (df["year_T"] == 2023)].dropna(subset=["gics_sector"])

ct = pd.crosstab(df["cluster_label"], df["gics_sector"])   # conteos
mat = ct.div(ct.sum(axis=1), axis=0)                        # P(sector | clúster)

UMBRAL = 0.80
filas_concentradas = []
filas_dispersas = []

for cluster in mat.index:
    fila = mat.loc[cluster]
    sector_dominante = fila.idxmax()       # sector con mayor proporción
    proporcion_max = fila.max()            # cuánta masa tiene ese sector
    # ¿cuántos sectores hacen falta para acumular el 90% de la masa? (medida de dispersión)
    masa_ordenada = fila.sort_values(ascending=False).cumsum()
    n_sectores_90 = int((masa_ordenada < 0.90).sum() + 1)
    n_empresas = int(ct.loc[cluster].sum())

    registro = {
        "cluster": cluster,
        "n_empresas": n_empresas,
        "sector_dominante": sector_dominante,
        "proporcion_dominante": round(proporcion_max, 3),
        "n_sectores_para_90pct": n_sectores_90,
    }
    if proporcion_max >= UMBRAL:
        filas_concentradas.append(registro)
    if n_sectores_90 >= 3:
        filas_dispersas.append(registro)

print(f"Filas concentradas (>={UMBRAL:.0%} en un solo sector): {len(filas_concentradas)} de {len(mat)}")
for r in filas_concentradas:
    print(f"  C{r['cluster']}: {r['proporcion_dominante']:.0%} en {r['sector_dominante']} "
          f"(n={r['n_empresas']})")

print(f"\nFilas dispersas (>=3 sectores para acumular 90%): {len(filas_dispersas)}")
for r in filas_dispersas:
    print(f"  C{r['cluster']}: dominante {r['sector_dominante']} "
          f"({r['proporcion_dominante']:.0%}), {r['n_sectores_para_90pct']} sectores para 90% "
          f"(n={r['n_empresas']})")

Filas concentradas (>=80% en un solo sector): 12 de 16
  C0: 91% en Communication Services (n=11)
  C1: 92% en Information Technology (n=13)
  C2: 100% en Industrials (n=8)
  C3: 100% en Real Estate (n=19)
  C4: 100% en Health Care (n=9)
  C5: 90% en Health Care (n=19)
  C6: 100% en Health Care (n=16)
  C7: 88% en Energy (n=16)
  C8: 100% en Utilities (n=18)
  C9: 100% en Financials (n=15)
  C11: 100% en Industrials (n=8)
  C12: 93% en Information Technology (n=15)

Filas dispersas (>=3 sectores para acumular 90%): 3
  C10: dominante Consumer Staples (71%), 3 sectores para 90% (n=24)
  C13: dominante Financials (54%), 5 sectores para 90% (n=41)
  C15: dominante Consumer Discretionary (36%), 4 sectores para 90% (n=14)
